# Fresh base-versus-LoRA instruction tuning

Two real optimizer steps by default, response-only loss, group-separated prompts and a fresh selected-adapter reload. No supplied predictions are used.

New project code prepared with AI assistance from concepts studied in the [IBM AI Engineering program](https://www.coursera.org/professional-certificates/ai-engineer). This is a separate experiment, not an IBM assessment submission. Data/model ownership remains with the original providers; data and weights are not bundled.

A local pilot is not Kaggle execution evidence or proof of a quality improvement.

## Configuration and dependencies

Attach the intended data and cache model weights, or explicitly enable downloads with internet access. Use the project's documented dependency environment. All training defaults to the bounded pilot; set `FULL_RUN=True` only for a separately budgeted complete run. `Save & Run All` must pass in the chosen Kaggle environment before publication.

In [ ]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256 = 'ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256 = '7bcf4c35919ee98a92bbb62ea3c573db73a0ae9d2e380145f4b19250ffe11cc6'

"""Compare a fixed OPT base with a genuinely trained, saved and reloaded LoRA adapter."""


import argparse
import gc
import hashlib
import json
import random
import time
from pathlib import Path

import torch



def prompt(record: dict) -> str:
    return f"### Instruction:\n{record['instruction'].strip()}\n\n### Response:\n"


def load_records(path: Path, seed: int) -> dict[str, list[dict]]:
    """Group normalized instructions before splitting; support JSON and JSONL sources."""
    text = path.read_text(encoding="utf-8")
    try:
        rows = json.loads(text)
    except json.JSONDecodeError:
        rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    groups = {}
    for row in rows:
        if (
            row.get("input", "").strip()
            or not row.get("instruction", "").strip()
            or not row.get("output", "").strip()
        ):
            continue
        key = " ".join(row["instruction"].split()).casefold()
        group_id = hashlib.sha256(key.encode()).hexdigest()
        record_id = hashlib.sha256(json.dumps(row, sort_keys=True).encode()).hexdigest()
        groups.setdefault(group_id, {})[record_id] = {**row, "id": record_id, "group": group_id}
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    if len(keys) < 10:
        raise ValueError("Need at least ten distinct instruction groups.")
    train_end, validation_end = int(0.8 * len(keys)), int(0.9 * len(keys))
    return {
        role: [row for key in selection for row in groups[key].values()]
        for role, selection in [
            ("train", keys[:train_end]),
            ("validation", keys[train_end:validation_end]),
            ("test", keys[validation_end:]),
        ]
    }


def encode_response(record: dict, tokenizer, max_length: int) -> dict:
    """Construct response labels explicitly; prompt and padded positions are excluded."""
    prefix = tokenizer.encode(prompt(record), add_special_tokens=True)
    answer = tokenizer.encode(record["output"].strip(), add_special_tokens=False)
    available = max_length - len(prefix)
    if available < 1 or not answer:
        raise ValueError("Prompt leaves no supervised response tokens.")
    truncated = len(answer) + 1 > available
    response = answer[:available] if truncated else answer + [tokenizer.eos_token_id]
    return {
        "input_ids": prefix + response,
        "labels": [-100] * len(prefix) + response,
        "response_truncated": truncated,
    }


def collate(examples: list[dict], pad_id: int) -> dict:
    """Mask padding by position, including when pad and EOS happen to share an ID."""
    width = max(len(row["input_ids"]) for row in examples)
    ids = torch.full((len(examples), width), pad_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for index, row in enumerate(examples):
        length = len(row["input_ids"])
        ids[index, :length] = torch.tensor(row["input_ids"])
        labels[index, :length] = torch.tensor(row["labels"])
        attention[index, :length] = 1
    return {"input_ids": ids, "attention_mask": attention, "labels": labels}


def response_nll(model, examples: list[dict], pad_id: int, device) -> dict:
    """Weight loss by actually supervised next-token targets."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(examples), 2):
            batch = {
                key: value.to(device)
                for key, value in collate(examples[start : start + 2], pad_id).items()
            }
            tokens = int((batch["labels"][:, 1:] != -100).sum())
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise ValueError("Non-finite response loss.")
            total += loss.item() * tokens
            count += tokens
    if not count:
        raise ValueError("No supervised evaluation tokens.")
    return {"nll": total / count, "tokens": count}


def generate_responses(
    model, tokenizer, records: list[dict], device, max_new_tokens: int
) -> list[str]:
    """Use the identical deterministic prompt/decoding contract for both models."""
    model.eval()
    results = []
    with torch.no_grad():
        for record in records:
            batch = tokenizer(prompt(record), return_tensors="pt")
            batch = {key: value.to(device) for key, value in batch.items()}
            output = model.generate(
                **batch,
                do_sample=False,
                num_beams=1,
                max_new_tokens=max_new_tokens,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
            results.append(
                tokenizer.decode(output[0, batch["input_ids"].shape[1] :], skip_special_tokens=True)
            )
    return results


def trainable_digest(model) -> str:
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            digest.update(name.encode())
            digest.update(parameter.detach().float().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def run(
    data_file: Path,
    output: Path,
    *,
    full: bool = False,
    allow_downloads: bool = False,
    seed: int = 42,
    epochs: int = 3,
    train_limit: int | None = None,
    evaluation_limit: int = 64,
) -> dict:
    """A pilot performs two real optimizer steps; no supplied predictions/logs are loaded."""
    if epochs < 1 or evaluation_limit < 1 or (train_limit is not None and train_limit < 1):
        raise ValueError("Epochs and record limits must be positive.")
    if not full and train_limit is not None:
        raise ValueError("Explicit training budgets require full=True; the pilot stays fixed.")
    from peft import LoraConfig, PeftModel, TaskType, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch.set_num_threads(1)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "facebook/opt-350m"
    model = AutoModelForCausalLM.from_pretrained(model_id, local_files_only=not allow_downloads).to(
        device
    )
    revision = getattr(model.config, "_commit_hash", None)
    if not revision:
        raise ValueError("The base model must have a resolved immutable revision.")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, local_files_only=not allow_downloads
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    partitions = load_records(data_file, seed)
    manifest = {
        role: [{"id": r["id"], "group": r["group"]} for r in rows]
        for role, rows in partitions.items()
    }
    write_json(output / "split-manifest.json", manifest)
    max_length = 256 if full else 128
    selected, encoded, rejected = {}, {}, {}
    for role, records in partitions.items():
        selected[role], encoded[role], rejected[role] = [], [], 0
        limit = (
            (train_limit if role == "train" else evaluation_limit)
            if full
            else (32 if role == "train" else 4)
        )
        for record in records:
            try:
                example = encode_response(record, tokenizer, max_length)
            except ValueError:
                rejected[role] += 1
                continue
            selected[role].append(record)
            encoded[role].append(example)
            if limit is not None and len(selected[role]) >= limit:
                break
        if not selected[role]:
            raise ValueError(f"No eligible {role} records.")
    started = time.perf_counter()
    base_nll = response_nll(model, encoded["test"], tokenizer.pad_token_id, device)
    base_outputs = generate_responses(
        model, tokenizer, selected["test"], device, 64 if full else 16
    )
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, config)
    before = trainable_digest(model)
    total_parameters = sum(p.numel() for p in model.parameters())
    parameters = [p for p in model.parameters() if p.requires_grad]
    trainable_parameters = sum(p.numel() for p in parameters)
    optimizer = torch.optim.AdamW(parameters, lr=2e-4)
    generator = torch.Generator().manual_seed(seed)
    history, seen = [], []
    best_loss, steps = float("inf"), 0
    adapter_dir = output / "best-adapter"
    for epoch in range(epochs if full else 1):
        model.train()
        order = torch.randperm(len(encoded["train"]), generator=generator).tolist()
        for start in range(0, len(order), 2):
            indices = order[start : start + 2]
            batch = {
                key: value.to(device)
                for key, value in collate(
                    [encoded["train"][i] for i in indices], tokenizer.pad_token_id
                ).items()
            }
            optimizer.zero_grad()
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise ValueError("Non-finite training loss.")
            loss.backward()
            optimizer.step()
            seen.extend(selected["train"][i]["id"] for i in indices)
            steps += 1
            if not full and steps >= 2:
                break
        score = response_nll(model, encoded["validation"], tokenizer.pad_token_id, device)
        history.append({"epoch": epoch + 1, "steps": steps, "validation_nll": score["nll"]})
        if score["nll"] < best_loss:
            best_loss = score["nll"]
            model.save_pretrained(adapter_dir, safe_serialization=True)
            tokenizer.save_pretrained(adapter_dir)
    changed = trainable_digest(model) != before
    if not changed:
        raise AssertionError("Optimizer steps did not change the adapter.")
    # Reload the validation-selected adapter, not the final epoch's parameters.
    model.load_adapter(adapter_dir, adapter_name="selected", is_trainable=False)
    model.set_adapter("selected")
    model.eval()
    probe = tokenizer(prompt(selected["validation"][0]), return_tensors="pt").to(device)
    with torch.no_grad():
        expected = model(**probe).logits[:, -1].cpu()
    del model, optimizer, parameters
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    fresh_base = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, local_files_only=not allow_downloads
    ).to(device)
    model = PeftModel.from_pretrained(fresh_base, adapter_dir).to(device).eval()
    with torch.no_grad():
        actual = model(**probe).logits[:, -1].cpu()
    torch.testing.assert_close(expected, actual, rtol=1e-5, atol=1e-5)
    adapter_nll = response_nll(model, encoded["test"], tokenizer.pad_token_id, device)
    adapter_outputs = generate_responses(
        model, tokenizer, selected["test"], device, 64 if full else 16
    )
    report = {
        "project": "instruction_tuning",
        "profile": ("budgeted-experiment" if train_limit is not None else "full")
        if full
        else "bounded-pilot",
        "budget": {
            "epochs": epochs if full else 1,
            "train_limit": train_limit,
            "evaluation_limit": evaluation_limit if full else 4,
        },
        "seed": seed,
        "base_model": model_id,
        "base_revision": revision,
        "data_sha256": file_sha256(data_file),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "complete_split_counts": {role: len(rows) for role, rows in partitions.items()},
        "selected_split_counts": {role: len(rows) for role, rows in selected.items()},
        "rejected_before_selection": rejected,
        "truncated_response_counts": {
            role: sum(row["response_truncated"] for row in examples)
            for role, examples in encoded.items()
        },
        "processed_training_ids": seen,
        "optimizer_steps": steps,
        "adapter_changed": changed,
        "trainable_parameters": trainable_parameters,
        "total_parameters": total_parameters,
        "adapter_sha256": file_sha256(adapter_dir / "adapter_model.safetensors"),
        "reload_max_abs_logit_difference": float((expected - actual).abs().max()),
        "reload_verified": True,
        "base_response_loss": base_nll,
        "adapter_response_loss": adapter_nll,
        "quality_claim": "Response NLL and paired generations only; task correctness and robust improvement remain unmeasured.",
        "decoding": {"do_sample": False, "num_beams": 1, "max_new_tokens": 64 if full else 16},
        "predictions": [
            {
                "id": r["id"],
                "instruction": r["instruction"],
                "reference": r["output"],
                "base": b,
                "adapter": a,
            }
            for r, b, a in zip(selected["test"], base_outputs, adapter_outputs, strict=True)
        ],
        "history": history,
        "seconds": time.perf_counter() - started,
        "runtime": runtime(["torch", "transformers", "peft"]),
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
    }
    write_json(output / "metrics.json", report)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-file", type=Path, required=True)
    parser.add_argument(
        "--output", type=Path, default=Path(".lab-cache/selected-projects/instruction_tuning")
    )
    parser.add_argument("--full", action="store_true")
    parser.add_argument("--allow-downloads", action="store_true")
    args = parser.parse_args()
    report = run(args.data_file, args.output, full=args.full, allow_downloads=args.allow_downloads)
    print({"steps": report["optimizer_steps"], "reload": report["reload_verified"]})



In [ ]:
# Keep this False for the small correctness pilot. Full training requires a separate run.
FULL_RUN = False
ALLOW_DOWNLOADS = False
SEED = 42

# Set this explicitly if multiple attached datasets match.
DATA_PATH = None
if DATA_PATH is None:
    if Path('/kaggle/input').is_dir():
        matches = sorted(Path('/kaggle/input').rglob('CodeAlpaca-20k.json'))
        if len(matches) != 1:
            raise ValueError('Set DATA_PATH to the intended attached dataset; expected one matching input.')
        DATA_PATH = matches[0]
    else:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'provenance/source-manifest.json').is_file()), None)
        if root is None:
            raise ValueError('Set DATA_PATH to the local data file/directory.')
        DATA_PATH = root / '.lab-cache/11-advanced-fine-tuning/instruction-tuning-with-llms/CodeAlpaca-20k.json'
DATA_PATH = Path(DATA_PATH)
root = locals().get('root') or Path.cwd()
OUTPUT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else root / '.lab-cache/selected-projects') / 'instruction_tuning'


In [ ]:
report = run(DATA_PATH, OUTPUT, full=FULL_RUN, seed=SEED, allow_downloads=ALLOW_DOWNLOADS)
print('Profile:', report['profile'])
print('Actual evaluated split sizes:', report['selected_split_counts'])

In [ ]:
print('Fresh adapter reload:', report['reload_verified'])
print('Response losses:', report['base_response_loss'], report['adapter_response_loss'])
for row in report['predictions']:
    print({key: row[key] for key in ['instruction', 'reference', 'base', 'adapter']})

## Interpretation

Inspect sample counts, split provenance and reload evidence before reading model scores. Pilot scores are too small to rank models or claim robust improvement. Full training, near-duplicate/source overlap review and task-specific quality evaluation remain separate checks.